# Testing 11 · Prostate IHC annotations from the step-1 intermediates

This loads the three saved step-1 parquets (legacy or versioned) and rebuilds the specimen annotations
used as outcomes by `text_interpretability/post_landmark/11_LR_for_IHC.ipynb` (`COMPASS/prostate_ihc/annotations.py`):
- `ihc_extraction/intermediate/prostate_cohort.parquet`: registry C61 or malignant prostate pathology patients (plus prostate OncoPanel patients in versioned extracts);
- `prostate_ihc_long.parquet`: one row per stain mention;
- `prostate_ihc_specimen_all.parquet`: one row per stained pathology part, with `prostate_relevant` (tiers 1–4).

Marker calls are recomputed per part from the lesional long rows (discordant NE marker = positive; discordant AR-axis marker = neither).
Flags: `ihc_any`, `ne_pos`, `nepc_like`, and `ar_low` (≥1 AR-axis marker negative, none positive, whatever the NE stains show).
No raw data are read and nothing is rerun or overwritten.


In [ ]:
import sys
from pathlib import Path
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'survival_common').is_dir()), None)
if ROOT is None: raise RuntimeError('Run from the repository or COMPASS/notebooks')
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from COMPASS.prostate_ihc import annotations as ann
from COMPASS.prostate_ihc import paths as ihc_paths

INTER_DIR = ihc_paths.default_paths().intermediate_dir   # env: COMPASS_IHC_EXTRACTION_ROOT
for f in ihc_paths.INTERMEDIATE_FILES:
    print(f"{INTER_DIR / f}  exists={(INTER_DIR / f).exists()}")


## Intermediates: counts and evidence tiers


In [ ]:
inter = ann.load_intermediates(INTER_DIR)
cohort, long, spec_all = inter["cohort"], inter["long"], inter["spec"]
print(f"{len(cohort):,} cohort patients; {len(long):,} stain rows "
      f"({int(long['target'].eq('lesional').sum()):,} lesional); {len(spec_all):,} stained parts")
display(spec_all.groupby("prostate_evidence").agg(specimens=("MRN_INT", "size"), patients=("MRN_INT", "nunique")))


## Annotations the LR notebook uses

Counts per flag among all stained parts and among prostate-relevant ones (`undated_*` = no specimen date, never an event), then the per-marker calls on prostate-relevant parts.

In [ ]:
calls = ann.marker_calls(long)
specimens = ann.specimen_annotations(long, spec_all)
display(ann.summary(specimens))
display(specimens[specimens["prostate_relevant"]].groupby("ihc_phenotype").size().rename("specimens").to_frame())
display(ann.marker_frequency(calls, specimens))